# Student Stress Level Prediction — Working ML Model
**MGM University · Institute of Information & Communication Technology · B.Tech Data Science · CA-III / CA4**

Team: Husen Bagwan (2407038) · Umar Shaikh (2407003) · Uzaif Shaikh (2407034)

This notebook implements the pipeline proposed in the project PPT and is organised to match the
**CA4 — 10 Marks** rubric:

| # | Parameter | Marks | Section below |
|---|---|---|---|
| 1 | Dataset Description & Understanding | 2 | Section 1 |
| 2 | Data Preprocessing & Preparation | 2 | Section 2 |
| 3 | ML Model Implementation | 2 | Section 3 |
| 4 | Model Performance & Evaluation | 2 | Section 4 |
| 5 | Live Demonstration, Team Contribution & Viva | 2 | Section 5 |

Run every cell top to bottom (Runtime → Run all, if using Google Colab). If `xgboost` is not
installed in your environment the notebook automatically falls back to scikit-learn's
`GradientBoostingClassifier`, which is conceptually the same gradient-boosted-trees family — install
xgboost with `!pip install xgboost` for the real thing (Colab already has it preinstalled).


In [ ]:
# If running on Google Colab or a fresh environment, uncomment the next line:
# !pip install -q xgboost

import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns

from sklearn.model_selection import train_test_split
from sklearn.preprocessing import StandardScaler, LabelEncoder
from sklearn.linear_model import LogisticRegression
from sklearn.tree import DecisionTreeClassifier, plot_tree
from sklearn.ensemble import RandomForestClassifier, GradientBoostingClassifier
from sklearn.metrics import (accuracy_score, precision_recall_fscore_support,
                              classification_report, confusion_matrix, ConfusionMatrixDisplay)

try:
    from xgboost import XGBClassifier
    HAS_XGB = True
except ImportError:
    HAS_XGB = False
    print("xgboost not found — will use GradientBoostingClassifier as a stand-in. "
          "Run '!pip install xgboost' and re-run this cell to use the real XGBoost.")

RANDOM_STATE = 42
sns.set_style("whitegrid")


## Section 1 — Dataset Description & Understanding *(2 Marks)*

**Source:** `student_lifestyle_dataset.csv` (team-collected / course-provided lifestyle survey data)
**Purpose:** predict a student's stress level from academic and lifestyle habits.


In [ ]:
df = pd.read_csv("student_lifestyle_dataset.csv")

print(f"Records (rows): {df.shape[0]}")
print(f"Features/columns: {df.shape[1]}")
print()
print("Column names and types:")
print(df.dtypes)


In [ ]:
df.head(10)


**Feature meanings**

| Column | Meaning | Type |
|---|---|---|
| Student_ID | Row identifier only — **not** used as a model feature | Numeric (ID) |
| Study_Hours_Per_Day | Hours spent studying per day | Numerical |
| Extracurricular_Hours_Per_Day | Hours in extracurricular activities per day | Numerical |
| Sleep_Hours_Per_Day | Hours of sleep per day | Numerical |
| Social_Hours_Per_Day | Hours of social interaction per day | Numerical |
| Physical_Activity_Hours_Per_Day | Hours of physical activity per day | Numerical |
| GPA | Current grade point average | Numerical |
| **Stress_Level** | **Target variable** — Low / Moderate / High | Categorical (3 classes) |


In [ ]:
df.describe().T


In [ ]:
print("Missing values per column:")
print(df.isnull().sum())
print()
print("Duplicate rows:", df.duplicated().sum())


In [ ]:
class_counts = df["Stress_Level"].value_counts()
class_pct = df["Stress_Level"].value_counts(normalize=True) * 100
print("Class distribution (counts):\n", class_counts)
print("\nClass distribution (%):\n", class_pct.round(2))

fig, ax = plt.subplots(figsize=(6, 4))
order = ["Low", "Moderate", "High"]
sns.countplot(x="Stress_Level", data=df, order=order, hue="Stress_Level", palette="viridis", legend=False, ax=ax)
ax.set_title("Stress Level — Class Distribution (2,000 records)")
for i, lvl in enumerate(order):
    ax.text(i, class_counts[lvl] + 15, f"{class_pct[lvl]:.1f}%", ha="center")
plt.tight_layout()
plt.savefig("class_distribution.png", dpi=150)
plt.show()


The dataset is **imbalanced**: High stress is ~3.5× more frequent than Low stress. Accuracy alone
can hide poor performance on the minority ("Low") class, so Section 4 also reports **per-class
precision, recall and F1**, not accuracy alone.


## Section 2 — Data Preprocessing & Preparation *(2 Marks)*

Steps performed:
1. Check missing values (none found above)
2. Check and remove duplicate rows
3. Drop `Student_ID` (identifier only, not predictive)
4. Encode the target label (`Stress_Level`) to numeric classes
5. Train/test split (80/20, **stratified** to preserve class proportions because of the imbalance)
6. Feature scaling with `StandardScaler` (needed for Logistic Regression; harmless for tree models)


In [ ]:
data = df.drop(columns=["Student_ID"]).drop_duplicates().reset_index(drop=True)
print("Shape after dropping ID and duplicates:", data.shape)

X = data.drop(columns=["Stress_Level"])
y = data["Stress_Level"]

le = LabelEncoder()
y_encoded = le.fit_transform(y)
print("Label mapping:", dict(zip(le.classes_, le.transform(le.classes_))))


In [ ]:
X_train, X_test, y_train, y_test = train_test_split(
    X, y_encoded, test_size=0.2, random_state=RANDOM_STATE, stratify=y_encoded
)
print("Train size:", X_train.shape, " Test size:", X_test.shape)

scaler = StandardScaler()
X_train_scaled = scaler.fit_transform(X_train)
X_test_scaled = scaler.transform(X_test)


In [ ]:
fig, axes = plt.subplots(2, 3, figsize=(15, 8))
features = X.columns.tolist()
order = ["Low", "Moderate", "High"]
for ax, feat in zip(axes.ravel(), features):
    sns.boxplot(x="Stress_Level", y=feat, data=data, order=order, hue="Stress_Level", palette="viridis", legend=False, ax=ax)
    ax.set_title(feat)
plt.tight_layout()
plt.savefig("features_vs_stress.png", dpi=150)
plt.show()


## Section 3 — ML Model Implementation *(2 Marks)*

Four classification models are trained on the same preprocessed data, as proposed in the PPT.

| Model | How it works | Why used |
|---|---|---|
| **Logistic Regression** | Weights each feature and turns the total into class probabilities (linear decision boundary) | Simple, interpretable baseline |
| **Decision Tree** | Splits the data step by step using if–else rules on features | Easy to visualize and explain |
| **Random Forest** | Many decision trees vote together, majority class wins | More stable, gives feature importance |
| **XGBoost** (or Gradient Boosting fallback) | Builds trees sequentially, each correcting the previous tree's errors | Captures complex/non-linear relationships |


In [ ]:
models = {
    "Logistic Regression": LogisticRegression(max_iter=1000, random_state=RANDOM_STATE),
    "Decision Tree": DecisionTreeClassifier(random_state=RANDOM_STATE),
    "Random Forest": RandomForestClassifier(n_estimators=300, random_state=RANDOM_STATE),
}
if HAS_XGB:
    models["XGBoost"] = XGBClassifier(eval_metric="mlogloss", random_state=RANDOM_STATE)
else:
    models["Gradient Boosting (XGBoost fallback)"] = GradientBoostingClassifier(random_state=RANDOM_STATE)

trained_models = {}
for name, model in models.items():
    model.fit(X_train_scaled, y_train)
    trained_models[name] = model
    print(f"Trained: {name}")


## Section 4 — Model Performance & Evaluation *(2 Marks)*

Metrics reported: **Accuracy, Precision, Recall, F1-score** (macro-averaged across the 3 classes)
and the **Confusion Matrix** for each model, as required for a classification project.


In [ ]:
results = []
all_preds = {}
for name, model in trained_models.items():
    preds = model.predict(X_test_scaled)
    all_preds[name] = preds
    acc = accuracy_score(y_test, preds)
    prec, rec, f1, _ = precision_recall_fscore_support(y_test, preds, average="macro", zero_division=0)
    results.append({"Model": name, "Accuracy": acc, "Precision (macro)": prec,
                     "Recall (macro)": rec, "F1-score (macro)": f1})

results_df = pd.DataFrame(results).set_index("Model").round(4)
results_df


In [ ]:
results_df.plot(kind="bar", figsize=(10, 5), colormap="viridis")
plt.title("Model Comparison — Test Set Metrics")
plt.ylabel("Score")
plt.xticks(rotation=20, ha="right")
plt.ylim(0, 1.05)
plt.legend(loc="lower right")
plt.tight_layout()
plt.savefig("model_comparison.png", dpi=150)
plt.show()


In [ ]:
fig, axes = plt.subplots(1, len(trained_models), figsize=(5 * len(trained_models), 4))
if len(trained_models) == 1:
    axes = [axes]
for ax, (name, preds) in zip(axes, all_preds.items()):
    cm = confusion_matrix(y_test, preds)
    disp = ConfusionMatrixDisplay(confusion_matrix=cm, display_labels=le.classes_)
    disp.plot(ax=ax, colorbar=False, cmap="Blues")
    ax.set_title(name, fontsize=10)
plt.tight_layout()
plt.savefig("confusion_matrices.png", dpi=150)
plt.show()


In [ ]:
for name, preds in all_preds.items():
    print(f"\n===== {name} — classification report =====")
    print(classification_report(y_test, preds, target_names=le.classes_, zero_division=0))


In [ ]:
best_model_name = results_df["F1-score (macro)"].idxmax()
print(f"Best model on this test set: {best_model_name} "
      f"(Macro-F1 = {results_df.loc[best_model_name, 'F1-score (macro)']:.4f})")

if best_model_name in ("Random Forest",):
    importances = pd.Series(trained_models[best_model_name].feature_importances_, index=X.columns)
    importances.sort_values().plot(kind="barh", figsize=(7, 4), color="teal")
    plt.title(f"Feature Importance — {best_model_name}")
    plt.tight_layout()
    plt.savefig("feature_importance.png", dpi=150)
    plt.show()


### Interpretation

- **Logistic Regression** gives a solid linear baseline but under-performs the tree-based models,
  which suggests the true relationship between lifestyle factors and stress is **not purely linear**
  (the earlier boxplots show sharp threshold-like separations rather than smooth linear trends).
- **Decision Tree / Random Forest / XGBoost** fit the non-linear thresholds in the data (e.g. a cutoff
  on `Study_Hours_Per_Day` combined with `Sleep_Hours_Per_Day`) far more closely, which is reflected in
  their higher accuracy and F1-scores — check your own run's numbers and, if a tree model reaches
  (near-)100% accuracy, say so explicitly in the report and note that this points to the labels in this
  dataset following a fairly clean rule on study/sleep hours rather than being noisy real-world
  measurements — a genuine finding worth discussing under "Limitations" (real student survey data
  would be noisier, and cross-validation / a larger external test set should be used before trusting
  this at face value).
- Because classes are imbalanced, the **macro-averaged** Precision/Recall/F1 (which weight all three
  classes equally) are reported instead of relying on accuracy alone, so that performance on the
  minority "Low" stress class is not masked by the majority "High" class.


## Section 5 — Live Demonstration, Team Contribution & Viva *(2 Marks)*

Use `predict_stress(...)` below to demo the trained model live during viva with any sample values.
Pick the model to demo with `DEMO_MODEL` (defaults to the best-performing model found above).


In [ ]:
DEMO_MODEL = best_model_name  # change to any key in trained_models, e.g. "Random Forest"
demo_clf = trained_models[DEMO_MODEL]

def predict_stress(study_hours, extracurricular_hours, sleep_hours,
                    social_hours, physical_activity_hours, gpa):
    """Takes a student's lifestyle inputs and returns the predicted stress level
    plus the model's class probabilities, using the DEMO_MODEL trained above."""
    row = pd.DataFrame([{
        "Study_Hours_Per_Day": study_hours,
        "Extracurricular_Hours_Per_Day": extracurricular_hours,
        "Sleep_Hours_Per_Day": sleep_hours,
        "Social_Hours_Per_Day": social_hours,
        "Physical_Activity_Hours_Per_Day": physical_activity_hours,
        "GPA": gpa,
    }])[X.columns]
    row_scaled = scaler.transform(row)
    pred_class = le.inverse_transform(demo_clf.predict(row_scaled))[0]
    if hasattr(demo_clf, "predict_proba"):
        proba = dict(zip(le.classes_, demo_clf.predict_proba(row_scaled)[0].round(3)))
    else:
        proba = None
    return pred_class, proba

# Example 1: heavy study load, little sleep -> expect higher stress
pred, proba = predict_stress(study_hours=8.5, extracurricular_hours=1.0, sleep_hours=5.0,
                              social_hours=1.5, physical_activity_hours=1.0, gpa=3.2)
print("Example 1 (high study, low sleep) ->", pred, proba)

# Example 2: balanced routine -> expect lower/moderate stress
pred, proba = predict_stress(study_hours=4.5, extracurricular_hours=3.0, sleep_hours=8.0,
                              social_hours=3.0, physical_activity_hours=4.0, gpa=3.0)
print("Example 2 (balanced routine) ->", pred, proba)

# Try your own values here for the live viva demo:
pred, proba = predict_stress(study_hours=6.0, extracurricular_hours=2.0, sleep_hours=7.0,
                              social_hours=2.0, physical_activity_hours=3.0, gpa=2.9)
print("Example 3 (your values) ->", pred, proba)


### Student-wise contribution (fill in for your viva, as listed in the CA4 rubric)

| Student | Responsibility |
|---|---|
| Student 1 | Dataset description & preprocessing (Sections 1–2) |
| Student 2 | ML model implementation (Section 3) |
| Student 3 | Model evaluation, results & overall demonstration (Sections 4–5) |

### Summary for the project report

- Copy the **metrics table** from Section 4 into the report's *4. Result & Analysis* table
  (Baseline / Proposed / % improvement columns can compare Logistic Regression as the baseline
  against the best tree-based model).
- Copy `class_distribution.png`, `confusion_matrices.png`, `model_comparison.png` and
  `feature_importance.png` as figures into *3. Methodology* and *4. Result & Analysis*.
- Use the Interpretation notes above for *4.1 Performance Analysis* and *5. Conclusion*.
